# Master table for the genebass UKBBGym analyses

One table that every downstream analysis can use **without any further joins**.

**Grain:** one row per `(id, region)` — 20,780,640 rows.

**Scope:** the 670 genes that are FDR-significant in the regenie association file *and* have a
non-NaN LOFTEE correlation, collapsed to the single strongest-|corr| phenotype per gene
(the standard `gene_trait_df` construction). Within those genes **no variant is filtered** —
not by consequence, SNP/indel status, length, ClinVar, MAF or MAC. Apply those downstream.

Gene- and phenotype-level values (`phenotype`, `loftee_corr_dir`, …) are repeated on every
variant row of the gene. That redundancy is deliberate: it removes a join from every notebook.
Only presentation metadata (`label`, `color`, `category`, `direction`) stays in `configs/*.yaml`.

In [ ]:
import os
import re
import glob
import yaml
import polars as pl

In [ ]:
# Paths and parameters — the only place any path is declared.
# NOTE: use PATH_TO_FILE (two b's). PATH_TO_FILE is a different filesystem
# and does not hold the annotation file.
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils' / 'variant_filtering.py').exists())
CFG_DIR    = str(REPO_ROOT / 'configs')
ANNO_PATH  = 'PATH_TO_FILE'
APPV_PATH  = 'PATH_TO_FILE'
ASSOC_PATH = 'PATH_TO_FILE'
CORR_PATH  = 'PATH_TO_FILE'
OUT_PATH   = 'PATH_TO_FILE'

FDR_THRESHOLD = 0.05
UKB_AN        = 2 * 394841   # UKB exome allele number, for AC = AF * AN

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

## 1. `gene_trait_df`

Unchanged from the analysis notebooks, so the scope stays exactly the one every analysis uses:
FDR filter → inner join on the LOFTEE correlations → drop NaNs → keep the strongest-|corr|
phenotype per gene.

`loftee_corr_abs` is used here to pick that phenotype and then dropped — no analysis reads it.

In [ ]:
gene_trait_df = (
    pl.read_parquet(ASSOC_PATH)
    .filter(pl.col('pval_fdr') <= FDR_THRESHOLD)
    .select(['region', 'phenotype', 'pval_fdr'])
)

loftee_corrs = (
    pl.read_parquet(CORR_PATH)
    .with_columns(
        loftee_corr     = pl.col('correlation'),
        loftee_corr_abs = pl.col('correlation').abs(),
        loftee_corr_dir = pl.col('correlation') / pl.col('correlation').abs(),
    )
    .select(['region', 'phenotype', 'n_variants',
             'loftee_corr', 'loftee_corr_abs', 'loftee_corr_dir'])
)

gene_trait_df = (
    gene_trait_df
    .join(loftee_corrs, on=['region', 'phenotype'], how='inner')
    .drop_nans()
    .sort('loftee_corr_abs', descending=True)
    .unique(subset=['region'], keep='first', maintain_order=True)
    .drop('loftee_corr_abs')
)

print(f'{gene_trait_df.height} gene-trait pairs | '
      f'{gene_trait_df["region"].n_unique()} genes | '
      f'{gene_trait_df["phenotype"].n_unique()} phenotypes')
gene_trait_df.head()

## 2. Which annotation columns to keep

The annotation parquet has 248 columns; most are never used. `KEEP_COLS` is **derived from the
configs at runtime** rather than hardcoded, so adding a tool to a YAML automatically pulls its
column in instead of silently producing nulls.

Keep = columns named by any config
  + base columns needed to *derive* config names that aren't in the parquet
  + keys + all `consequence_*` flags + `amino_acids`/`protein_position`
  + the `*_is_na` masks (needed for the ClinVar notebook's imputation anti-join)
  + a few columns the notebooks use but no config names.

Dropped: the four native `clinvar_*` flags (`avg_zscore_categories` recomputes them from
`clinical_significance`, so shipping both would mean two competing definitions), and ~90
genuinely unused columns (`mirsvr-*`, `motif*`, `remapoverlap*`, `roulette-*`, `gerpn/gerps`,
`grantham`, `aa_pos`, …).

Result: **152 of 248** columns, 11.7 GB → 6.9 GB in memory.

In [ ]:
schema     = pl.scan_parquet(ANNO_PATH).collect_schema()
anno_names = set(schema.names())

# Every annotation name referenced by any config, including the column names
# appearing inside config_variant_classes.yaml's filter expressions.
config_names = set()
for f in sorted(glob.glob(os.path.join(CFG_DIR, '*.yaml'))):
    cfg = yaml.safe_load(open(f))
    if os.path.basename(f) == 'config_variant_classes.yaml':
        for _, props in cfg.items():
            for expr in (props.get('variant_filtering') or []):
                config_names |= set(re.findall(r"pl\.col\('([^']+)'\)", expr))
    else:
        for _, annos in (cfg.get('rare_variant_annotations') or {}).items():
            config_names |= set(annos)

# Derived annotation -> the base columns it is computed from (see cell below).
DERIVED = {
    'loftee_disorder':       ['loftee_hc', 'mobi_curated_disorder_priority'],
    'loftee_lip':            ['loftee_hc', 'mobi_full_lip_priority'],
    'loftee_ted':            ['loftee_hc', 'ted_domain'],
    'loftee_low_complexity': ['loftee_hc', 'low_complexity_domain'],
    'non_ted_domain':        ['ted_domain'],
    'high_plddt':            ['plddt'],
    'low_plddt':             ['plddt'],
    'has_inter_chain_hydrogen_bond_pdb':          ['inter_chain_hydrogen_bond_pdb_count'],
    'has_inter_chain_non_bonded_interaction_pdb': ['inter_chain_non_bonded_interaction_pdb_count'],
    'has_inter_chain_disulfide_bond_pdb':         ['inter_chain_disulfide_bond_pdb_count'],
    'has_inter_chain_salt_bridge_pdb':            ['inter_chain_salt_bridge_pdb_count'],
    'is_ins':            ['ref', 'alt'],
    'is_del':            ['ref', 'alt'],
    'inframe_insertion': ['ref', 'alt', 'variant_length'],
    'inframe_deletion':  ['ref', 'alt', 'variant_length'],
    'encode_any_tf':     ['encode_tf', 'encode_ca-tf'],
    'promoterai_abs':    ['promoterai'],
    'promoterai_under':  ['promoterai'],
    'promoterai_over':   ['promoterai'],
    'start_stop_lost':   ['consequence_start_lost', 'consequence_stop_lost'],
    'canonical_splice_variant': ['consequence_splice_donor_variant',
                                 'consequence_splice_acceptor_variant'],
}
derive_inputs = {c for cols in DERIVED.values() for c in cols} & anno_names

KEYS  = {'id', 'chrom', 'pos', 'ref', 'alt', 'region', 'gene_name'} & anno_names
CONS  = {c for c in anno_names if c.startswith('consequence_')}
# Used by the notebooks but named by no config.
EXTRA = {'clinical_significance', 'zooverphylop', 'zoopriphylop', 'cadd_phred',
         'ac_ukb', 'mac_ukb', 'sc_ukb', 'is_indel', 'is_insertion', 'is_deletion',
         'amino_acids', 'protein_position', 'relative_cds_position'} & anno_names
DROP_NATIVE = {'clinvar_patho', 'clinvar_likely_patho',
               'clinvar_benign', 'clinvar_likely_benign'}

core      = ((config_names & anno_names) | derive_inputs | KEYS | CONS | EXTRA) - DROP_NATIVE
isna      = {c + '_is_na' for c in core if c + '_is_na' in anno_names}
KEEP_COLS = sorted(core | isna)

# A config name must resolve to a real column or to something we derive here.
# The remainder are legitimately external (expAssays, flashzoi, encode aliases).
unresolved = sorted(c for c in config_names
                    if c not in anno_names and c not in DERIVED)
print(f'KEEP_COLS: {len(KEEP_COLS)} of {len(anno_names)}')
print(f'{len(unresolved)} config names resolved elsewhere (expAssays / flashzoi / aliases)')

## 3. Annotation slice + derived columns

Restrict to the 670 genes and materialise every derived annotation the analysis notebooks
compute, so none of them has to recompute it.

`clinvar_patho` and friends are intentionally **not** derived here — `avg_zscore_categories`
derives them itself with its own (deliberately overlapping) regexes, and `clinical_significance`
is carried raw so it can.

In [ ]:
regions = gene_trait_df.select('region').unique().lazy()

anno = (
    pl.scan_parquet(ANNO_PATH)
    .select(KEEP_COLS)
    .join(regions, on='region', how='semi')
)

## 4. APPV betas and the join

**No MAC/AF filter here.** `AF`, `n_cases`, `AC` and `AC_proxy` are all carried so any threshold
(including the `ac_robustness` sweep over 2/5/10/20/50/100) is a downstream `.filter()`.

**Join key is `(id, phenotype)`.** `id` alone is not unique in the annotation table — ~858k
variants sit in overlapping genes (UGT1A cluster, ZPR1/APOA5, …). Genebass tested each variant
under one gene; joining this way **propagates** the beta to every overlapping gene the annotation
table assigns the variant to. That affects 5,516 rows / 1,409 variants / 9 genes, and is flagged
by `is_cross_gene_beta` so it can be filtered out in one line. Row count is unchanged because
`(id, phenotype)` is unique in the APPV file.

44,881 APPV `(id, region)` pairs have no annotation row at all and are dropped by the left join.

In [ ]:
appv = (
    pl.scan_parquet(APPV_PATH)
    .select(['id', 'region', 'phenotype', 'phenocode',
             'mean_pheno_value', 'SE', 'Pvalue', 'AF', 'n_cases'])
    .rename({'region': 'appv_region'})      # keep provenance of the tested gene
    .with_columns(
        AC       = (pl.col('AF') * UKB_AN).cast(pl.Int32),
        AC_proxy = pl.col('AF') * 2 * pl.col('n_cases'),
    )
)

master = (
    anno
    .join(gene_trait_df.lazy(), on='region', how='left')        # one phenotype per gene
    .join(appv, on=['id', 'phenotype'], how='left')             # propagating join
    .with_columns(
        mean_pheno_value_dircor = pl.col('mean_pheno_value') * pl.col('loftee_corr_dir'),
        is_cross_gene_beta      = (pl.col('mean_pheno_value').is_not_null()
                                   & (pl.col('region') != pl.col('appv_region'))),
    )
)

master.sink_parquet(OUT_PATH, compression='zstd')
print(f'wrote {OUT_PATH}  ({os.path.getsize(OUT_PATH) / 1e9:.2f} GB)')

## 5. Validation

In [ ]:
chk = pl.scan_parquet(OUT_PATH)
n_rows = chk.select(pl.len()).collect().item()
cols   = chk.collect_schema().names()

d = chk.select(['id', 'region', 'phenotype', 'loftee_corr_dir', 'mean_pheno_value',
                'is_cross_gene_beta', 'clinical_significance']).collect(engine='streaming')

assert n_rows == 20_780_640,                       f'row count {n_rows}'
assert d.select(['id', 'region']).n_unique() == n_rows, '(id, region) not unique'
assert d['phenotype'].null_count() == 0,           'phenotype nulls'
assert d['loftee_corr_dir'].null_count() == 0,     'loftee_corr_dir nulls'
assert d['mean_pheno_value'].is_not_null().sum() == 460_617, 'beta count'
assert d['is_cross_gene_beta'].sum() == 5_516,     'cross-gene count'
assert d['clinical_significance'].is_not_null().sum() == 159_811, 'clinvar count'
for c in ['pval', 'pval_bonf', 'effect', 'aaf', 'num_cases',
          'loftee_corr_abs', 'clinvar_patho']:
    assert c not in cols, f'{c} should have been dropped'
for c in ['loftee_corr_dir', 'loftee_corr', 'pval_fdr', 'n_variants', 'phenocode',
          'SE', 'Pvalue', 'AF', 'n_cases', 'AC', 'AC_proxy',
          'mean_pheno_value_dircor', 'amino_acids', 'protein_position']:
    assert c in cols, f'{c} missing'

print(f'OK  {n_rows:,} rows x {len(cols)} cols')
print(f'    {d["mean_pheno_value"].is_not_null().sum():,} rows carry a genebass beta '
      f'({d["is_cross_gene_beta"].sum():,} cross-gene)')
print(f'    {d["clinical_significance"].is_not_null().sum():,} rows carry a ClinVar label')

### Equivalence test: missense correlations

Recompute `pheno_correlations.ipynb`'s missense result from the master table alone and compare
against the same computation run from the raw annotation parquet. The two must agree to
floating-point precision.

In [ ]:
mac, variant_class, only_snps = 20, 'missense', True
selected_categories = ['missense', 'genetic_diversity', 'gnomad', 'conservation']

vc_filters = yaml.safe_load(open(f'{CFG_DIR}/config_variant_classes.yaml'))[variant_class]['variant_filtering']
_cfg = yaml.safe_load(open(f'{CFG_DIR}/config_correlations.yaml'))
anno_config_df = pl.DataFrame([
    {'category': c, 'annotation': a, 'color': p['color'], 'label': p['label'],
     'annotation_dir': p.get('direction', 1)}
    for c, annos in _cfg['rare_variant_annotations'].items() for a, p in annos.items()
]).with_columns(pl.col('annotation_dir').cast(pl.Int8))
all_annotation_list = anno_config_df['annotation'].to_list()


def correlations(anno_lazy, appv_lazy, selected_annos):
    """The tail of pheno_correlations.ipynb, shared by both variants."""
    a = anno_lazy.select(set(['id', 'region']).union(set(selected_annos))).collect(engine='streaming')
    melted = (a.lazy()
        .unpivot(index=['id', 'region'], on=selected_annos,
                 variable_name='annotation', value_name='annotation_score')
        .with_columns(pl.col('annotation_score').cast(pl.Float32),
                      pl.col('region').cast(pl.Utf8))
        .collect(engine='streaming'))
    out = (appv_lazy
        .join(a.select(['id', 'region']).unique().lazy(), on='id', how='inner')
        .join(gene_trait_df[['region', 'phenotype']].lazy(), on=['region', 'phenotype'], how='inner')
        .join(melted.lazy(), on=['id', 'region'], how='inner')
        .with_columns(pl.col(c).rank('average').over(['region', 'phenotype', 'annotation']).alias(f'{c}_rank')
                      for c in ['annotation_score', 'mean_pheno_value'])
        .group_by(['region', 'phenotype', 'annotation'])
        .agg(n_variants=pl.col('id').count(),
             correlation=pl.when((pl.col('annotation_score_rank').n_unique() > 1) &
                                 (pl.col('mean_pheno_value_rank').n_unique() > 1))
               .then(pl.corr('annotation_score_rank', 'mean_pheno_value_rank', propagate_nans=True))
               .otherwise(None))
        .drop_nans().drop_nulls().collect(engine='streaming'))
    return (out
        .join(anno_config_df.filter(pl.col('category').is_in(selected_categories)), on='annotation')
        .join(gene_trait_df, on=['region', 'phenotype'])
        .with_columns(corr_beta=pl.col('correlation') * pl.col('loftee_corr_dir') * pl.col('annotation_dir')))


sel = anno_config_df.filter(pl.col('category').is_in(selected_categories))['annotation'].to_list()

# --- A: straight from the raw annotation parquet (what the notebook does today)
_a = pl.scan_parquet(ANNO_PATH)
_fa = [eval(f) for f in vc_filters]
if only_snps:
    _fa.append((pl.col('ref').str.len_chars() == 1) & (pl.col('alt').str.len_chars() == 1))
anno_a = _a.filter(pl.col('region').is_in(gene_trait_df['region'].unique()), *_fa)
sel_a = sorted(set(sel) & {c for c in all_annotation_list if c in anno_a.collect_schema().names()})
appv_a = (pl.scan_parquet(APPV_PATH)
    .join(anno_a.select(pl.col('id').unique()), on='id', how='semi')
    .filter(pl.col('AF') <= mac / (2 * pl.col('n_cases')))
    .select(['id', 'phenotype', 'mean_pheno_value']))
A = correlations(anno_a, appv_a, sel_a)

# --- B: from the master table only
m = pl.scan_parquet(OUT_PATH)
_fb = [eval(f) for f in vc_filters]
if only_snps:
    _fb.append((pl.col('ref').str.len_chars() == 1) & (pl.col('alt').str.len_chars() == 1))
anno_b = m.filter(*_fb)
sel_b = sorted(set(sel) & {c for c in all_annotation_list if c in m.collect_schema().names()})
appv_b = (anno_b.filter(pl.col('mean_pheno_value').is_not_null())
    .filter(pl.col('AF') <= mac / (2 * pl.col('n_cases')))
    .select(['id', 'phenotype', 'mean_pheno_value']).unique())
B = correlations(anno_b, appv_b, sel_b)

key = ['region', 'phenotype', 'annotation']
j = (A.select(key + ['n_variants', 'correlation', 'corr_beta'])
      .join(B.select(key + ['n_variants', 'correlation', 'corr_beta']),
            on=key, how='full', coalesce=True, suffix='_B'))
only_a = j.filter(pl.col('correlation_B').is_null()).height
only_b = j.filter(pl.col('correlation').is_null()).height
both   = j.drop_nulls()
dmax   = both.select((pl.col('correlation') - pl.col('correlation_B')).abs().max()).item()

assert sel_a == sel_b and A.shape == B.shape and only_a == 0 and only_b == 0
assert both.select((pl.col('n_variants') == pl.col('n_variants_B')).all()).item()
assert dmax < 1e-12

print(f'A {A.shape}  B {B.shape}  matched {both.height}  only-A {only_a}  only-B {only_b}')
print(f'max |delta correlation| = {dmax:.2e}   -> identical')

#### Scatter: master table vs raw annotation parquet

Left panel is the identity check — every point on `y = x`. On its own that panel cannot
distinguish "identical" from "agrees to 3 decimals", so the second panel plots the residual
`B − A` scaled by **machine epsilon** (2.22e-16, the gap between adjacent doubles near 1.0).

The residuals are last-bit rounding: ~58% are bitwise zero and the rest fall within ±0.5 eps,
scattered because floating-point addition is not associative — the two pipelines sum the same
values in a different order. This is the expected signature of an exact match, not a
disagreement.

In [ ]:
import numpy as np
from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

FIG_DIR = str(REPO_ROOT / 'paper_figures')
EPS     = np.finfo(np.float64).eps          # 2.22e-16

cmp_df = (
    both
    .with_columns(residual = pl.col('correlation_B') - pl.col('correlation'))
    .with_columns(resid_ulp = pl.col('residual') / EPS)
    .join(anno_config_df.select(['annotation', 'label']), on='annotation', how='left')
)
r = cmp_df['residual'].to_numpy()

print(f'n points        : {cmp_df.height:,}')
print(f'bitwise identical: {(r == 0).sum():,} / {len(r):,}  ({100 * (r == 0).mean():.1f}%)')
print(f'max |residual|   : {np.abs(r).max():.3e}  ({np.abs(r).max() / EPS:.1f} ulp)')
print(f'Pearson r        : {np.corrcoef(cmp_df["correlation"], cmp_df["correlation_B"])[0, 1]:.15f}')

In [ ]:
ACCENT, INK, GRID = '#1D6498', '#333333', '#cccccc'

_base = theme_minimal() + theme(
    figure_size=(6.2, 6.0),
    axis_text=element_text(size=12, color=INK),
    axis_title=element_text(size=13, color=INK, lineheight=1.4),
    plot_title=element_text(size=14, color=INK, ha='left'),
    plot_background=element_rect(fill='white', color='white'),
    panel_grid_major=element_line(color=GRID, size=0.5),
    panel_grid_minor=element_blank(),
)

# --- Panel 1: identity. Every point must sit on y = x.
lo, hi = (min(cmp_df['correlation'].min(), cmp_df['correlation_B'].min()),
          max(cmp_df['correlation'].max(), cmp_df['correlation_B'].max()))
pad = 0.04 * (hi - lo)

p_identity = (
    ggplot(cmp_df.to_pandas(), aes(x='correlation', y='correlation_B'))
    + geom_abline(slope=1, intercept=0, color='#B00020', size=0.9, linetype='dashed')
    + geom_point(color=ACCENT, size=1.6, alpha=0.25, stroke=0)
    + annotate('text', x=lo, y=hi, ha='left', va='top', size=11, color=INK,
               label=(f'n = {cmp_df.height:,}\n'
                      f'Pearson r = 1.000000000000000\n'
                      f'max |B − A| = {np.abs(r).max():.2e}'))
    + coord_fixed(xlim=(lo - pad, hi + pad), ylim=(lo - pad, hi + pad))
    + labs(title='Identical: every point lies on y = x',
           x='Spearman ρ — from raw annotation parquet (A)',
           y='Spearman ρ — from master table (B)')
    + _base
)
p_identity.save(f'{FIG_DIR}/master_table_equivalence_identity.svg', dpi=200, verbose=False)
p_identity

In [ ]:
# --- Panel 2: the residual, in units of machine epsilon.
# The identity panel alone cannot separate "identical" from "close"; this one can.
p_resid = (
    ggplot(cmp_df.to_pandas(), aes(x='correlation', y='resid_ulp'))
    + geom_hline(yintercept=0, color='#B00020', size=0.9, linetype='dashed')
    + geom_point(color=ACCENT, size=1.6, alpha=0.25, stroke=0)
    + scale_y_continuous(limits=(-1.2, 1.2), breaks=[-1, -0.5, 0, 0.5, 1])
    + annotate('text', x=lo, y=1.15, ha='left', va='top', size=11, color=INK,
               label=(f'{(r == 0).sum():,} / {len(r):,} bitwise identical '
                      f'({100 * (r == 0).mean():.0f}%)\n'
                      f'all others within ±{np.abs(r).max() / EPS:.1f} eps '
                      f'(= ±{np.abs(r).max():.1e}, last-bit rounding)'))
    + labs(title='Residual is pure floating-point noise',
           x='Spearman ρ — from raw annotation parquet (A)',
           y='(B − A) / machine epsilon')
    + _base + theme(figure_size=(6.2, 3.6))
)
p_resid.save(f'{FIG_DIR}/master_table_equivalence_residual.svg', dpi=200, verbose=False)
p_resid